# 01. API 기초 — 요청 하나, 응답 하나

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/01_api_basics.ipynb)

> 🧭 **스토리** — 한빛테크 신입 AI 담당자인 당신의 첫 과제. "AI를 붙이려면 API부터 알아야지." 출장비 환율을 묻는 API를 불러 보고, OpenAI를 SDK 없이 직접 부르고, 마지막에는 **출장비 계산 API를 직접 만들어** 본다.
>
> 📖 **이론 연결** — 이론1 p8~17 (API 기초: 요청·응답, AI API 지도, SDK vs HTTP, API 만들기, 왜 지금 쉬운가) · ⏱ **10분** · 🖥 **CPU**
>
> GPU가 필요 없습니다. 기본 런타임(CPU)으로 실행하세요.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
HTTP 요청 라이브러리 `requests`, OpenAI SDK, 그리고 API를 직접 만들 `fastapi`·`uvicorn`을 설치합니다. 코랩에 이미 있는 것은 건너뜁니다.

In [ ]:
%pip -q install openai requests fastapi uvicorn

## 1. API = 주소에 요청하고 JSON으로 응답받기 🔵
해외 출장비는 달러로 정해져 있고 원화 환산은 출장 시작일의 환율로 합니다(규정 제8조). 키 없이 쓰는 공개 환율 API [Frankfurter](https://frankfurter.dev)(유럽중앙은행 기준환율)에 **오늘 1달러가 몇 원인지** 물어봅니다.

In [ ]:
import requests
r = requests.get("https://api.frankfurter.dev/v1/latest", params={"from": "USD", "to": "KRW"}, timeout=10)
print("요청 주소 :", r.url)
print("상태 코드 :", r.status_code)
print("헤더 일부 :", {k: r.headers.get(k) for k in ["content-type", "date"]})
data = r.json()                               # 본문(JSON 문자열) → 파이썬 dict
print("본문(JSON):", data)
print(f"\n→ {data['date']} 기준 1 USD = {data['rates']['KRW']:,.2f} KRW")

# 일부러 틀려 보기 — 없는 주소를 부르면?
bad = requests.get("https://api.frankfurter.dev/v1/no_such_path", timeout=10)
print("\n[틀린 주소] 상태 코드:", bad.status_code, "· 오류 본문:", bad.json())

💡 **해석** — 강사 실측(2026-10-13 호출): `200`, `content-type: application/json`, 본문 `{"amount": 1.0, "base": "USD", "date": "2026-10-12", "rates": {"KRW": 1333.09}}`. 즉 **1 USD = 1,333.09원**(유럽중앙은행 10월 12일 기준환율)이었습니다. 틀린 주소는 `404`와 `{"status": 404, "message": "not found"}`였습니다.
- **요청** = 메서드(`GET`/`POST`) + 주소(URL, `?from=USD&to=KRW` 같은 파라미터 포함) + 헤더 + (본문)
- **응답** = 상태 코드 + 헤더 + 본문(대부분 JSON). `r.json()` 한 줄로 파이썬 dict가 됩니다.
- 자주 만나는 상태 코드: **200** 성공 · **400/422** 요청 형식 오류 · **401** 인증 실패(키 문제) · **404** 주소 없음 · **429** 요청이 너무 많음 · **500번대** 서버 문제. 오류도 JSON으로 오므로 **코드로 읽어서 처리**할 수 있습니다.

## 2. OpenAI를 SDK 없이 직접 부르기 🔵
AI API도 1절과 똑같은 **HTTP 요청**입니다. 다른 점은 두 가지뿐입니다. ① `POST`로 본문(JSON)을 보낸다. ② 헤더에 **키**를 넣는다. 먼저 키를 입력합니다(⚙️).

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
# 입력한 키는 모델 조회 API(GET /v1/models/{모델} = client.models.retrieve)로 바로 확인합니다. 토큰을 쓰지 않아 비용이 없습니다.
import os, json, urllib.request, urllib.error
from getpass import getpass
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.

def check_key(key: str):
    """키 확인 → (상태, 안내). 키 문자열과 오류 본문은 출력하지 않는다(유형·상태 코드만)."""
    req = urllib.request.Request(f"https://api.openai.com/v1/models/{MODEL_API}",
                                 headers={"Authorization": f"Bearer {key}"})
    try:
        with urllib.request.urlopen(req, timeout=15):
            return "ok", "키 확인 완료"
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode()).get("error", {})        # 판단에만 쓰고 출력하지 않는다
        except Exception:
            err = {}
        code, msg = err.get("code"), str(err.get("message", "")).lower()
        scope = "insufficient permissions" in msg or "missing scopes" in msg   # 권한 범위가 좁은 제한 키
        if e.code == 401 and not scope and code in ("invalid_api_key", None):
            return "bad", "키가 올바르지 않습니다(401). 빠진 글자·앞뒤 공백 없이 다시 붙여 넣으세요."
        if e.code in (401, 403):
            return "ok", f"키는 유효하지만 모델 조회 권한이 없습니다({e.code}). 실습은 진행하고, 호출이 막히면 강사에게 알려 주세요."
        if e.code == 404:
            return "ok", f"키는 유효하지만 모델 {MODEL_API}에 접근할 수 없습니다(404). 강사에게 알려 주세요."
        if e.code == 429:
            return "ok", "키는 유효합니다. 요청 한도에 걸렸습니다(429). 잠시 뒤 다시 실행하세요."
        return "err", f"확인 실패: HTTPError {e.code}. 잠시 뒤 이 셀을 다시 실행하세요."
    except Exception as e:
        return "err", f"확인 실패: {type(e).__name__}. 네트워크 연결을 확인하고 이 셀을 다시 실행하세요."

key = os.environ.get("OPENAI_API_KEY", "").strip()
for attempt in range(1, 4):
    if not key:
        key = getpass(f"OpenAI API 키 ({attempt}/3): ").strip()
    status, msg = check_key(key)
    print(msg)
    if status != "bad":
        break
    key = ""                                   # 틀린 키는 버리고 다시 입력받는다
if status == "bad":
    os.environ.pop("OPENAI_API_KEY", None)
    raise RuntimeError("키 확인 3회 실패: 강사에게 키를 다시 받은 뒤 이 셀을 다시 실행하세요.")
os.environ["OPENAI_API_KEY"] = key
print("모델:", MODEL_API)

In [ ]:
import os
try:
    r = requests.post(
        "https://api.openai.com/v1/responses",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",   # 키 = 신분증
                 "Content-Type": "application/json"},
        json={"model": MODEL_API, "input": "한빛테크 신입 AI 담당자로서 한 줄로 자기소개해 줘.",
              "reasoning": {"effort": "none"}},
        timeout=60)
    print("상태 코드:", r.status_code)
    body = r.json()
    if r.status_code == 200:
        msg = [o for o in body["output"] if o["type"] == "message"][0]   # output 배열에서 메시지 항목 찾기
        print("답:", msg["content"][0]["text"])
        print("usage:", {k: body["usage"][k] for k in ["input_tokens", "output_tokens"]})
    else:
        # 오류 본문의 message 는 키 일부를 되풀이할 수 있어 출력하지 않습니다
        print("키가 없거나 틀리면 401(인증 실패)이 옵니다. 오류 종류:",
              body.get("error", {}).get("type"), "/", body.get("error", {}).get("code"))
except Exception as e:
    print("요청 실패(네트워크 확인):", type(e).__name__)

💡 **해석** — 헤더의 `Authorization: Bearer <키>`가 **신분증**입니다. 서버는 이 키로 누가 부르는지 확인하고 요금을 청구합니다. 그래서 키는 코드에 적지 않고 출력하지도 않습니다.
응답 JSON은 `output`(배열) → 메시지 항목 → `content[0]["text"]`로 내려가야 답이 나옵니다. 사용량 `usage.input_tokens`·`output_tokens`가 곧 요금의 단위입니다(02번 노트북). Anthropic·Google·네이버 CLOVA 등 **모든 AI API가 이 모양**입니다. 주소·헤더 이름·JSON 필드 이름만 조금씩 다릅니다.

## 3. 같은 호출을 SDK로 🔵
OpenAI 파이썬 SDK를 쓰면 2절이 세 줄이 됩니다. JSON을 파고들 필요 없이 `output_text`로 답을 꺼냅니다.

In [ ]:
from openai import OpenAI
client = OpenAI()                      # 환경변수 OPENAI_API_KEY 를 알아서 읽습니다
try:
    resp = client.responses.create(model=MODEL_API, input="한 줄로 자기소개해 줘.", reasoning={"effort": "none"})
    print("답:", resp.output_text)
    print("usage:", resp.usage.input_tokens, "/", resp.usage.output_tokens)
except Exception as e:
    print("API 호출 실패(키·네트워크 확인):", type(e).__name__, getattr(e, "status_code", ""))

# SDK가 대신 해 주는 것
from openai.types.responses import Response
print("\n--- SDK가 대신 해 주는 것 ---")
print("자동 재시도 횟수 :", client.max_retries)        # 429·500번대·연결 오류면 기다렸다가 다시 보냄
print("타임아웃        :", client.timeout)
print("요청 주소(base) :", client.base_url)
print("응답 타입       :", Response, "→ Pydantic 모델:", Response.__mro__[1].__name__)
print("응답 필드 일부  :", list(Response.model_fields)[:8])

💡 **해석** — SDK = **HTTP 호출 + 자동 재시도(기본 2회) + 타임아웃 + 타입(Pydantic 모델) + 스트리밍** 포장입니다. 응답이 dict가 아니라 타입이 있는 객체라서 `resp.usage.output_tokens`처럼 점으로 꺼내고, 오타는 편집기가 잡아 줍니다(2절은 `body["usage"]["output_tokens"]`).
SDK가 부르는 주소는 `base_url`(`https://api.openai.com/v1/`)입니다. **이 값만 바꾸면** 같은 SDK로 로컬 Ollama·네이버 CLOVA Studio를 부릅니다. 05번 노트북에서 직접 합니다.

## 4. 내 API 만들기 — FastAPI 🔵
이제 반대편입니다. 한빛테크 **국내 출장비 계산 API**를 만들어 봅니다. 규정 표 1의 일비·숙박비 상한으로 **일비 × 일수 + 숙박비 × 박수**를 돌려줍니다(식비·교통비는 생략).
- `TripRequest`(Pydantic 모델) = 이 API가 받는 **요청 본문의 모양**
- `@app.post("/trip_cost")` = 이 주소로 `POST`가 오면 아래 함수를 실행
- 마지막 부분은 이 앱을 코랩 안에서 **백그라운드 서버**로 띄웁니다.

In [ ]:
from typing import Literal
from fastapi import FastAPI
from pydantic import BaseModel, Field

RATES = {"사원": (20_000, 80_000), "과장": (30_000, 110_000), "부장": (40_000, 150_000)}   # (일비/일, 숙박비 상한/박) — 규정 표 1

class TripRequest(BaseModel):
    grade: Literal["사원", "과장", "부장"]   # 직급: 셋 중 하나만 허용 (아니면 422)
    nights: int = Field(ge=0)                # 박: 0 이상
    days: int = Field(gt=0)                  # 일: 0 초과

app = FastAPI(title="한빛테크 출장비 API")

@app.post("/trip_cost")
def trip_cost(req: TripRequest):
    daily, hotel = RATES[req.grade]          # grade는 이미 검증됐으므로 바로 조회
    breakdown = {"일비": daily * req.days, "숙박비": hotel * req.nights}
    return {"total": sum(breakdown.values()), "breakdown": breakdown}

# 코랩 안에서 서버를 백그라운드 스레드로 띄웁니다 (실제 서비스라면 서버 컴퓨터에서 `uvicorn main:app`)
import threading, socket, time, uvicorn

def port_free(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0

if not globals().get("_SERVER_STARTED"):
    PORT = 8000 if port_free(8000) else 8001
    threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"),
                     daemon=True).start()
    _SERVER_STARTED = True
BASE = f"http://127.0.0.1:{PORT}"
for _ in range(20):                                   # 최대 10초 기다리기
    try:
        requests.get(BASE + "/openapi.json", timeout=1); break
    except Exception:
        time.sleep(0.5)
print("내 API 서버:", BASE)

### 내 API 부르기 + 자동 문서 🔵
1절에서 환율 API를 부른 것과 **똑같은 방법**으로 부릅니다. 이어서 FastAPI가 코드에서 자동으로 만든 **API 설명서(OpenAPI)** `/openapi.json`도 받아 봅니다.

In [ ]:
import json
r = requests.post(BASE + "/trip_cost", json={"grade": "과장", "nights": 2, "days": 3})
print("상태 코드:", r.status_code, "· 응답:", r.json())

spec = requests.get(BASE + "/openapi.json").json()
print("\n경로 목록:", list(spec["paths"]))
print("TripRequest 스키마:")
print(json.dumps(spec["components"]["schemas"]["TripRequest"], ensure_ascii=False, indent=2))

💡 **해석**
- 응답은 `{"total": 310000, "breakdown": {"일비": 90000, "숙박비": 220000}}`입니다. 과장 2박 3일은 30,000 × 3 + 110,000 × 2 = **310,000원**입니다. 함수 하나에 데코레이터 한 줄을 붙였을 뿐인데 **누구나 HTTP로 부를 수 있는 API**가 되었습니다.
- `TripRequest` 클래스는 `{"type": "object", "properties": {"grade": {"type": "string", "enum": ["사원", "과장", "부장"]}, "nights": {"type": "integer", "minimum": 0}, …}, "required": ["grade", "nights", "days"]}`라는 **JSON Schema**가 되었습니다. 13번 노트북의 툴 콜링에서 모델에게 주는 **도구 스키마가 바로 이 JSON Schema**입니다. 사람이 읽는 설명서이면서 모델이 읽는 도구 설명서도 됩니다. (코랩 밖에서 서버를 띄우면 브라우저의 `/docs`에서 같은 내용을 화면으로 볼 수 있습니다.)

## 5. 바꿔서 실행 🟡 — 잘못된 요청 보내기
`BAD`의 값을 바꿔 보내 보세요. 예: `nights`를 `"이틀"`(글자) → `"2"`(숫자 모양 글자) → 아예 빼기, `grade`를 `"인턴"`으로.

In [ ]:
BAD = {"grade": "과장", "nights": "이틀", "days": 3}          # ← 바꿔 보세요
r = requests.post(BASE + "/trip_cost", json=BAD)
print("상태 코드:", r.status_code)
body = r.json()
if r.status_code == 422:                                   # Pydantic 검증 실패
    d = body["detail"][0]
    print("어디가:", d["loc"], "| 왜:", d["msg"])
else:
    print("응답:", body)

💡 **해석** — `"이틀"`을 보내면 함수가 실행되기도 전에 **422**와 함께 `loc: ['body', 'nights']`, `msg: 'Input should be a valid integer, unable to parse string as an integer'`가 돌아옵니다. `"2"`는 숫자로 바꿀 수 있어 통과하고, 아예 빼면 422 `'Field required'`가 오고, `"인턴"`은 `Literal` 검사에 걸려 역시 **422** `"Input should be '사원', '과장' or '부장'"`이 옵니다(`days: 0`도 `Field(gt=0)`에 걸려 422). **검증은 서버가 하고, 호출자는 무엇이 틀렸는지 명확한 오류를 받습니다.** OpenAI API가 잘못된 요청에 400을 돌려주는 것도 같은 원리입니다.

## 6. 모델도 API다 🔵
OpenAI의 모델은 OpenAI 서버 안에 있고, 우리는 2·3절처럼 HTTP로 부릅니다. 그렇다면 **모델을 우리 컴퓨터에 두고 API로 열 수도** 있지 않을까요?
- **Ollama**는 `ollama serve`를 실행하면 `http://localhost:11434/v1/chat/completions`에 **OpenAI와 같은 모양의 API**를 엽니다. 4절에서 우리가 `/trip_cost`를 연 것과 같은 일을, 계산 함수 대신 **언어 모델**로 하는 것입니다.
- 그래서 3절의 SDK에서 `base_url`만 `http://localhost:11434/v1`로 바꾸면 같은 코드로 로컬 모델을 부릅니다.
- 05번 노트북에서 Ollama를 설치하고 정확히 이것을 합니다.

### 🤔 생각해보기

① 13번 노트북에서 에이전트가 쓰는 계산 도구(`calc`, 파이썬 함수)를 이 `/trip_cost` **API로 바꾸면** 무엇이 달라질까요?

② **프로바이더 API**(OpenAI)와 **내 API**(`/trip_cost`)의 차이는 결국 무엇일까요?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

**①**
- **네트워크**: 함수 호출은 즉시 끝나지만 API는 네트워크를 거칩니다. 지연·타임아웃·재시도가 생깁니다.
- **공유**: 한 번 띄우면 여러 에이전트·여러 팀·다른 언어(자바스크립트 등)가 같은 계산을 씁니다. 규정이 바뀌면 서버 한 곳만 고치면 됩니다.
- **버전**: 요청 모양(스키마)이 바뀌면 부르는 쪽이 깨질 수 있습니다. `/v1/trip_cost`처럼 버전을 붙여 관리합니다.
- **권한**: 아무나 부르면 안 되는 API(급여·인사 정보)에는 2절처럼 **키·인증**을 붙여야 합니다.
- 스키마(`/openapi.json`)가 이미 있으므로 **도구 설명서를 따로 쓸 필요가 줄어듭니다.**

**②** 모양은 같습니다. 주소·메서드·헤더·JSON 본문·상태 코드는 모두 같습니다. 차이는 **서버 안에서 무엇이 답을 만드느냐**입니다. 내 API는 우리가 쓴 계산 코드가 답을 만들고, 프로바이더 API는 **거대한 언어 모델**이 답을 만듭니다. 그래서 결과가 매번 같지 않을 수 있고, 토큰 단위로 요금이 붙고, 키가 필요합니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- API = 주소에 요청(메서드·헤더·본문)을 보내고 응답(상태 코드·헤더·JSON 본문)을 받는 약속이다.
- AI API도 같은 HTTP 요청이다. 헤더의 Bearer 키가 신분증이고, 401은 키 문제다. 키는 출력하지 않는다.
- SDK는 HTTP 호출에 재시도·타임아웃·타입·스트리밍을 씌운 포장이다. `base_url`만 바꾸면 다른 서버를 부른다.
- FastAPI로 함수 하나를 API로 만들 수 있다. Pydantic 모델이 요청 검증과 JSON Schema 문서를 함께 만든다.
- 모델도 API다. Ollama는 로컬 모델을 OpenAI와 같은 모양의 API로 연다(05번).

> 다음 노트북: **02_tokenizer — 토큰이 곧 비용이다**